In [ ]:
import sys
import os
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Current device: {device}")

_repo = os.environ.get("NWP_BENCHMARK_ROOT", os.getcwd())
if _repo not in sys.path:
    sys.path.insert(0, _repo)

from tc_paths import ibtracs_csv_path
from load_IBTrACS import BestTracker

tracker = BestTracker(ibtracs_csv_path())

Current device: cpu


In [ ]:


import os
import datetime  # for datetime handling
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
from geopy.distance import geodesic  # Haversine distance on the sphere
from ghr.utils.s3_client import s3_client
# try:
#     from nwp.datasets.s3_client import s3_client
# except:
#     s3_client = None
    
# Clear proxy environment variables
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

s3 = s3_client('h')

def get_max_wind_near_center(wind_speed, lat, lon, center_lat, center_lon, search_radius_deg=1.0):
    """
    Return max wind speed within a lat/lon box around the TC center.
    
    Args:
        wind_speed (np.ndarray): wind speed data (2D)。
        lat (np.ndarray): latitude array。
        lon (np.ndarray): longitude array。
        center_lat (float): TC center latitude。
        center_lon (float): TC center longitude。
        search_radius_deg (float): search radius in degrees。
        
    Returns:
        float: max wind speed near the TC center。
    """
    # Find lat/lon indices within the search box
    lat_indices = np.where((lat >= center_lat - search_radius_deg) & (lat <= center_lat + search_radius_deg))[0]
    lon_indices = np.where((lon >= center_lon - search_radius_deg) & (lon <= center_lon + search_radius_deg))[0]

    # Restrict wind speed to the search region
    wind_speed_in_range = wind_speed[np.ix_(lat_indices, lon_indices)]

    # Return max wind speed in the search region
    max_wind_speed = np.max(wind_speed_in_range)
    return max_wind_speed

def find_typhoon_center(msl, lat, lon, init_lat, init_lon, search_radius_km=278):
    """
    Find TC center as minimum MSL within a search radius.
    
    Args:
        msl (np.ndarray): mean sea level pressure (2D)。
        lat (np.ndarray): latitude array。
        lon (np.ndarray): longitude array。
        init_lat (float): initial latitude。
        init_lon (float): initial longitude。
        search_radius_km (float): search radius in km (default 278 km)。
        
    Returns:
        tuple: TC center latitude and longitude。
    """
    # Constants
    EARTH_RADIUS_KM = 6371.0  # Earth radius in km
    DEG_PER_KM_LAT = 1 / 111.32  # latitude degrees per km
    
    # Longitude degrees per km from initial latitude
    deg_per_km_lon = 1 / (111.32 * np.cos(np.radians(init_lat)))
    
    # Convert search radius km to lat/lon deltas
    lat_range = search_radius_km * DEG_PER_KM_LAT
    lon_range = search_radius_km * deg_per_km_lon
    
    # Clip lat/lon search bounds
    lat_min, lat_max = init_lat - lat_range, init_lat + lat_range
    lon_min, lon_max = init_lon - lon_range, init_lon + lon_range

    # Indices within the search region
    lat_indices = np.where((lat >= lat_min) & (lat <= lat_max))[0]
    lon_indices = np.where((lon >= lon_min) & (lon <= lon_max))[0]

    # Restrict MSL to the search region
    msl_in_range = msl[np.ix_(lat_indices, lon_indices)]
    lat_in_range = lat[lat_indices]
    lon_in_range = lon[lon_indices]

    # Location of minimum pressure in the search region
    min_idx = np.unravel_index(np.argmin(msl_in_range), msl_in_range.shape)
    center_lat = lat_in_range[min_idx[0]]
    center_lon = lon_in_range[min_idx[1]]

    return center_lat, center_lon
    # Indices within the search region
    lat_indices = np.where((lat >= lat_min) & (lat <= lat_max))[0]
    lon_indices = np.where((lon >= lon_min) & (lon <= lon_max))[0]

    # Restrict MSL to the search region
    msl_in_range = msl[np.ix_(lat_indices, lon_indices)]
    lat_in_range = lat[lat_indices]
    lon_in_range = lon[lon_indices]

    # Location of minimum pressure in the search region
    min_idx = np.unravel_index(np.argmin(msl_in_range), msl_in_range.shape)
    center_lat = lat_in_range[min_idx[0]]
    center_lon = lon_in_range[min_idx[1]]

    return center_lat, center_lon

def track_typhoon(member_files, init_lat, init_lon, variable_name="msl", 
                  pressure_threshold=101200, wind_speed_threshold=10.2, 
                  search_radius_km=278, max_life_time_steps=40, 
                  lat_min=-50, lat_max=50, lon_min=0, lon_max=360, 
                  wind_radius_km=100):
    """
    Track one ensemble member with pressure/wind/distance/lifetime cutoffs.
    
    Args:
        member_files (list): list of NetCDF paths for the member。
        init_lat (float): initial TC center latitude。
        init_lon (float): initial TC center longitude。
        variable_name (str): variable for TC detection (default "msl")。
        pressure_threshold (float): pressure cutoff threshold (hPa)。
        wind_speed_threshold (float): wind speed cutoff threshold (m/s)。
        max_distance_threshold (float): max consecutive TC displacement (km)。
        max_life_time_steps (int): max tracking time steps。
        lat_min, lat_max, lon_min, lon_max (float): geographic bounds。
        wind_radius_km (float): radius for wind computation (km)。
        
    Returns:
        list: TC track as [(lat, lon, time), ...]。
    """
    trajectory = []  # accumulated TC track
    current_lat, current_lon = init_lat, init_lon  # initial position

    for member_file in member_files:
        if  os.path.exists(member_file): 
            ds = xr.open_dataset(member_file)
        else:
            ds = s3.read_nc_from_BytesIO(member_file)

        lat = ds['latitude'].values
        lon = ds['longitude'].values
        msl = ds[variable_name].values.squeeze()
        u10 = ds['u10'].values.squeeze()  # 10 m u wind component
        v10 = ds['v10'].values.squeeze()  # 10 m v wind component

        try:
            time = ds['time'].values[0]
        except:
            time = ds['time'].values
        # Locate TC center
        center_lat, center_lon = find_typhoon_center(msl, lat, lon, current_lat, current_lon, search_radius_km=278)

        # Read pressure at center
        center_idx_lat = np.argmin(np.abs(lat - center_lat))
        center_idx_lon = np.argmin(np.abs(lon - center_lon))
        center_pressure = msl[center_idx_lat, center_idx_lon]

        # Wind speed magnitude
        wind_speed = np.sqrt(u10**2 + v10**2)

        # Max wind near center
        wind_max = get_max_wind_near_center(wind_speed, lat, lon, center_lat, center_lon, wind_radius_km)

        # Displacement from previous step
        distance = geodesic((current_lat, current_lon), (center_lat, center_lon)).km

        # Apply termination criteria
        if (
            center_pressure > pressure_threshold or
            wind_max < wind_speed_threshold or
            distance > search_radius_km or
            len(trajectory) > max_life_time_steps or
            not (lat_min <= center_lat <= lat_max and lon_min <= center_lon <= lon_max)
        ):
            print(f"Tracking stopped: Pressure={center_pressure:.2f}, Max Wind Speed={wind_max:.2f}, "
                  f"Distance={distance:.2f}, Steps={len(trajectory)}")
            break

        # Advance center and append to track
        current_lat, current_lon = center_lat, center_lon
        trajectory.append((center_lat, center_lon, time))

    return trajectory


def plot_typhoon_tracks(trajectories, output_path=None, extent=None):
    """
    Plot TC tracks for multiple members plus the mean track.
    
    Args:
        trajectories (dict): member tracks as {name: [(lat, lon, time), ...]}。
        output_path (str): output image path; if None, show interactively。
        extent (list): map extent [leftlon, rightlon, lowerlat, upperlat]。
    """
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Set map extent
    if extent:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Add map features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    
    # Accumulator for mean track
    mean_trajectory = {}
    
    # Plot each member track
    for member_name, trajectory in trajectories.items():
        if len(trajectory) > 0:  # skip empty tracks
            lats, lons, times = zip(*trajectory)  # unpack track as lats, lons, times
            # if member_name=="M 0":
            ax.plot(lons, lats, marker='o', markersize=3, linestyle='--', label=member_name, alpha=0.6)  # connect track points

            # Add points to mean-track accumulator
            if member_name != 'best':
                for idx, (lat, lon, t) in enumerate(trajectory):
                    if t not in mean_trajectory: # initialize the time 
                        mean_trajectory[t] = {"lat_sum": 0, "lon_sum": 0, "count": 0}
                    mean_trajectory[t]["lat_sum"] += lat  # sum latitude
                    mean_trajectory[t]["lon_sum"] += lon  # sum longitude
                    mean_trajectory[t]["count"] += 1  # member count for this time
    
    # Build mean track
    mean_lats, mean_lons, mean_times = [], [], []
    for t, values in sorted(mean_trajectory.items()):  # sorted by time
        # if values["count"]<10:
        #     break
        mean_lats.append(values["lat_sum"] / values["count"])  # mean latitude
        mean_lons.append(values["lon_sum"] / values["count"])  # mean longitude
        mean_times.append(t)  # time label
    
    # Plot mean track
    if mean_lats and mean_lons:
        ax.plot(mean_lons, mean_lats, color='black', linewidth=1,  marker='^', markersize=4, label='Mean Track')  # bold black mean track

    # Legend (multi-column)
    ax.legend(loc='upper left', fontsize=10, title='Members',framealpha=0.5, ncol=4)  # legend columns
    
    # Title
    ax.set_title("Typhoon Tracks Across Members", fontsize=16)

    # Save or show figure
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {output_path}")
    else:
        plt.show()


if __name__ == "__main__":
    # Example member file path lists
    timestamp = '2024-11-04T00:00:00'  # start time
    # root = f'/mnt/petrelfs/hantao.dispatch/NWP/ai-nwp-deploy/data/output/era5/{timestamp}_1step_100iter'
    root = f'/mnt/petrelfs/hantao.dispatch/NWP/ai-nwp-deploy/data/output/era5'
    root = f'/mnt/petrelfs/hantao.dispatch/NWP/ai-nwp-deploy/data/output/analysis'
    root = f'ai4earth/nwp_predictions/FengWu_GHR_0.25_ENS/2024'
    # root = f'nwp_predictions/FengWu_GHR/2024'
    ens = True
    # root = f"/home/hantao-dispatch/NWP/ai-nwp-deploy/FengWu-GHR-ENS/output/2024"
    # User-specified initial TC center
    init_lat, init_lon = 12.8, 131.8  # initial lat/lon

    # Forecast lead configuration
    lead_times = 10  # forecast length in days (10)
    interval_hours = 6  # 6-hour interval
    # Store tracks per member
    trajectories = {}

    # Build timestamp per lead time
    base_time = datetime.datetime.strptime(timestamp, "%Y-%m-%dT%H:%M:%S")
    all_forecast_times = [
        (base_time + datetime.timedelta(hours=i * interval_hours)).strftime("%Y-%m-%dT%H:%M:%S")
        for i in range(0, lead_times * 24 // interval_hours)
    ]

    # ----------------------
    # User-specified time window
    # ----------------------
    start_time = "2024-11-04 00:00:00"  # start time
    end_time = "2024-11-14 23:59:59"    # end time
    best_tc = tracker.get_typhoon_path(
        name="Talim", #。YINXING
        season=2023,
        start_time=start_time,
        end_time=end_time,
        time_step=6,
    )
    print(best_tc)
    
    # output path
    output_path = "./demos/typhoon_tracks.png"

    # map extent [leftlon, rightlon, lowerlat, upperlat]
    extent = [85, 140, 5, 30]

    # Plot all member tracks
    best = []
    for forecast_time in all_forecast_times:
        row = best_tc[best_tc["ISO_TIME"] == forecast_time]
        if not row.empty:
            best.append((row["LAT"].values, row["LON"].values,forecast_time))
    trajectories['best']= best

    
    # Loop over members
    for member_idx in range(0, 10):  # example: 50 members
        member_name = f"M {member_idx}"
        print(f"Processing {member_name}")

        if ens:
            member_files = [
            f"{root}/{timestamp}/{forecast_time}_{member_idx}.nc"
            for forecast_time in all_forecast_times
            ]
        else:
            member_files = [
            f"{root}/{timestamp}/{forecast_time}.nc"
            for forecast_time in all_forecast_times
            ]
        
        print(member_files)
        # Track this member
        trajectories[member_name] = track_typhoon(member_files,  init_lat, init_lon, 
                                                  pressure_threshold=101300, 
                                                  search_radius_km=350,
                                                  variable_name="msl",
                                                  )
    os.environ['http_proxy'] = 'http://127.0.0.1:8889/'
    os.environ['https_proxy'] = 'http://127.0.0.1:8889/'
    plot_typhoon_tracks(trajectories, output_path=output_path, extent=extent)
    
    
    


No data found for typhoon 'Talim' in the specified time range.
Empty DataFrame
Columns: []
Index: []


/mnt/shared-storage-user/hantao-dispatch/NWP/ai-nwp-deploy/evaluation/load_IBTrACS.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  typhoon_data['ISO_TIME'] = pd.to_datetime(typhoon_data['ISO_TIME'], errors='coerce')


KeyError: 'ISO_TIME'